# Cleaning 1: the project's rules (baseline)

**Input:** `data/team_data_full_v1.duckdb` (download it into `data/`). **Output:** `data/clean/florian_01_project_rules.parquet`:
one row per field-season, yield in t/ha, fold and 2 km gap columns, and Sentinel-2 features by season month,
ready for a Ridge model.

**Rationale.** These are the rules the project used for all results so far, written out one by one, so that the
other cleaning notebooks can change one rule at a time and we can see which change helps:

1. Field point or polygon: a village point or a rounded point (about 160 m off) cannot be matched to 10 m pixels.
2. Maize only: other crops have other seasons and yields.
3. A measured yield (also a measured zero).
4. Zimbabwe 2024 recorded zeros out: their share rises with the visit month (13 % in March, 100 % in July), so most
   are fields harvested before the visit, not crop failures.
5. At most 15 t/ha (above is not plausible for smallholder maize).
6. Not irrigated (rain-fed maize only).
7. Out: 3 rows whose yield does not follow the provider's own formula.

Then the lock: sealed test fields (fold 0 and 2 km around it) are dropped here, so no later step can touch them.
Features: the clearest Sentinel-2 pass of each month of this season (surface reflectance) and NDVI. To choose
the picture of each month in another way, use every pass in `raw_v1.v_reflectance` / `raw_oaf5.v_reflectance`
(bands, `scl`, `cld`, date). Cloudy months stay
empty (NaN): fill them inside the cross-validation, not here.

In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve().parents[1]                 # this notebook lives in notebook/<name>/
sys.path.insert(0, str(REPO / "scripts"))
from spatial_folds import add_folds, train_test

DATABASE = next(REPO / "data" / n for n in ("team_data_full_v1.duckdb", "team_data_v1.duckdb")
                if (REPO / "data" / n).exists())
OUT = REPO / "data" / "clean" / "florian_01_project_rules.parquet"
con = duckdb.connect(str(DATABASE), read_only=True)
labels = con.sql("select * from labels").df()
print(f"{len(labels):,} field-seasons in {labels.data_set.nunique()} data sets")

95,114 field-seasons in 12 data sets


## 1. The rules, one by one (the first rule that fits removes a row)

In [2]:
rules = {
    "no field point": ~labels.location_quality.isin(["field_point", "field_polygon"]),
    "not maize": labels.crop != "maize",
    "no measured yield": ~labels.label_status.isin(["measured_positive", "recorded_zero", "verified_crop_failure"])
                         | labels.yield_t_ha.isna(),
    "Zimbabwe 2024 recorded zero": (labels.data_set == "zimbabwe_2024") & (labels.label_status == "recorded_zero"),
    "above 15 t/ha": labels.yield_t_ha > 15,
    "irrigated": labels.irrigated.fillna(False).astype(bool),
    "provider formula mismatch": labels.exclusion_reason == "yield_formula_mismatch",
}
left = pd.Series(True, index=labels.index)
removed = {}
for name, hit in rules.items():
    removed[name] = (left & hit).groupby(labels.data_set).sum()
    left &= ~hit
steps = pd.DataFrame(removed)
steps.insert(0, "start", labels.groupby("data_set").size())
steps["kept"] = left.groupby(labels.data_set).sum()
steps.loc["total"] = steps.sum()
assert (left == labels.keep).all(), "these rules must give the project's keep flag"
steps

,start,no field point,not maize,no measured yield,Zimbabwe 2024 recorded zero,above 15 t/ha,irrigated,provider formula mismatch,kept
data_set,,,,,,,,,
burundi_oaf,24460,20784,2624,0,0,0,0,0,1052
kenya_oaf,24063,904,0,0,0,0,0,0,23159
malawi_2022,529,0,0,0,0,0,0,0,529
malawi_2023,267,0,0,0,0,0,0,0,267
malawi_oaf_2020,2594,762,0,94,0,0,0,0,1738
nigeria_oaf,115,115,0,0,0,0,0,0,0
rwanda_oaf,24964,20004,3683,24,0,3,0,0,1250
tanzania_oaf,7949,5974,0,0,0,0,0,0,1975
uganda_oaf,246,0,0,0,0,0,0,0,246


## 2. Folds and the lock (sealed rows are dropped here)

In [3]:
clean = add_folds(labels.loc[left, ["field_season_id", "release", "data_set", "season_year", "season",
                                      "yield_t_ha", "label_status", "strict"]], con)
print("sealed rows dropped:", clean.groupby("data_set").locked.sum().to_dict())
clean = clean[~clean.locked].reset_index(drop=True)               # locked is now false everywhere; train_test needs it
print(f"{len(clean):,} open rows; fold sizes:", {int(k): int(v) for k, v in clean.fold.value_counts().sort_index().items()})

sealed rows dropped: {'burundi_oaf': 220, 'kenya_oaf': 7831, 'malawi_2022': 113, 'malawi_2023': 62, 'malawi_oaf_2020': 348, 'rwanda_oaf': 250, 'tanzania_oaf': 395, 'uganda_oaf': 50, 'zambia_oaf': 260, 'zimbabwe_2024': 514, 'zimbabwe_lacuna': 541}
25,648 open rows; fold sizes: {1: 6291, 2: 6604, 3: 6379, 4: 6374}


## 3. Sentinel-2 features: the clearest pass of each season month (this season, clear months only)

In [4]:
bands = ["b02", "b03", "b04", "b05", "b06", "b07", "b08", "b8a", "b11", "b12"]
monthly = con.sql(f"""select field_season_id, season_month, {", ".join(bands)}
                       from monthly_clearest where which_year = 'this_season' and clear""").df()
monthly = monthly[monthly.field_season_id.isin(clean.field_season_id)]
monthly["ndvi"] = (monthly.b08 - monthly.b04) / (monthly.b08 + monthly.b04)
wide = monthly.pivot(index="field_season_id", columns="season_month", values=bands + ["ndvi"])
wide.columns = [f"{band}_m{month}" for band, month in wide.columns]
clean = clean.merge(wide, left_on="field_season_id", right_index=True, how="left")
clean["n_clear_months"] = clean.field_season_id.map(monthly.groupby("field_season_id").size()).fillna(0).astype(int)
features = list(wide.columns)
print(f"{len(features)} feature columns; share empty by month:")
clean[[f"ndvi_m{m}" for m in range(1, 10)]].isna().mean().round(2).to_frame("empty").T

99 feature columns; share empty by month:


,ndvi_m1,ndvi_m2,ndvi_m3,ndvi_m4,ndvi_m5,ndvi_m6,ndvi_m7,ndvi_m8,ndvi_m9
empty,0.02,0.02,0.07,0.07,0.06,0.02,0.04,0.08,0.68


## 4. Save the clean table

In [5]:
OUT.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(OUT, index=False)
print(f"saved {OUT.relative_to(REPO).as_posix()}: {clean.shape[0]:,} rows x {clean.shape[1]} columns")
summary = clean.groupby("data_set").agg(rows=("field_season_id", "size"), strict=("strict", "sum"),
                                        median_clear_months=("n_clear_months", "median"))
summary

saved data/clean/florian_01_project_rules.parquet: 25,648 rows x 118 columns


,rows,strict,median_clear_months
data_set,,,
burundi_oaf,832,105,6.0
kenya_oaf,15328,5126,8.0
malawi_2022,416,416,9.0
malawi_2023,205,205,8.0
malawi_oaf_2020,1390,0,9.0
rwanda_oaf,1000,676,6.0
tanzania_oaf,1580,1405,9.0
uganda_oaf,196,91,8.0
zambia_oaf,1037,1027,9.0


## How to use it for Ridge

```python
table = pd.read_parquet(OUT)
for k in (1, 2, 3, 4):
    train, test = train_test(table, k)     # train leaves out the 2 km gap around fold k
    # fit on table[train], score on table[test]; fill empty months from the training rows only
```
Score each data set separately (and Burundi, Uganda only in pools). Compare cleaning notebooks on the same folds.

In [6]:
for k in (1, 2, 3, 4):
    train, test = train_test(clean, k)
    print(f"fold {k}: train {train.sum():,}  test {test.sum():,}  left out by the 2 km gap {int(clean[f'gap_{k}'].sum()):,}")

fold 1: train 17,583  test 6,291  left out by the 2 km gap 1,774
fold 2: train 17,258  test 6,604  left out by the 2 km gap 1,786
fold 3: train 17,262  test 6,379  left out by the 2 km gap 2,007
fold 4: train 17,497  test 6,374  left out by the 2 km gap 1,777
